In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from catboost import CatBoostRegressor

In [2]:
df = pd.read_csv("clean_agri_dataset.csv")

df.columns = df.columns.str.strip().str.lower()

print("Original Shape :", df.shape)

# Remove duplicates
df.drop_duplicates(inplace=True)

# Remove missing values
df.dropna(inplace=True)

# Remove unrealistic price outliers
df = df[df["price_per_quintal"] <= 20000]

# Recalculate profit
df["profit"] = (
    df["price_per_quintal"]
    - df["cost_per_quintal"]
)

print("Final Shape :", df.shape)

df.head()

Original Shape : (6379, 13)
Final Shape : (6377, 13)


,state,market,crop,month,year,cost_per_quintal,price_per_quintal,season,profit,crop_avg_price,market_avg_price,season_avg_price,month_avg_price
0,Maharashtra,Bhusaval,Potato,10,2024,2500,2800,Kharif,300,1491.619529,1819.200000,1729.608710,1769.549839
1,Maharashtra,Kamthi,Onion,3,2024,1500,2500,Rabi,1000,1053.422507,2559.523810,1359.969900,872.285992
2,Maharashtra,Junnar(Otur),Potato,5,2025,500,1920,Zaid,1420,1491.619529,971.698113,1025.862269,1020.443966
3,Maharashtra,Shrirampur,Onion,6,2023,200,1450,Kharif,1250,1053.422507,1414.060000,1729.608710,1463.281646
4,Maharashtra,Phaltan,Wheat,6,2023,2350,3200,Kharif,850,2377.829060,1160.714286,1729.608710,1463.281646


In [3]:
df["crop_avg_price"] = (
    df.groupby("crop")["price_per_quintal"]
      .transform("mean")
)

df["market_avg_price"] = (
    df.groupby("market")["price_per_quintal"]
      .transform("mean")
)

df["season_avg_price"] = (
    df.groupby("season")["price_per_quintal"]
      .transform("mean")
)

df["month_avg_price"] = (
    df.groupby("month")["price_per_quintal"]
      .transform("mean")
)

df.to_csv("clean_agri_dataset.csv", index=False)

print(df.columns.tolist())

['state', 'market', 'crop', 'month', 'year', 'cost_per_quintal', 'price_per_quintal', 'season', 'profit', 'crop_avg_price', 'market_avg_price', 'season_avg_price', 'month_avg_price']


In [4]:
TARGET = "price_per_quintal"

X = df.drop(columns=[TARGET, "profit"])
y = df[TARGET]

categorical_features = [
    "state",
    "market",
    "crop",
    "season"
]

numeric_features = [
    "month",
    "year",
    "cost_per_quintal",
    "crop_avg_price",
    "market_avg_price",
    "season_avg_price",
    "month_avg_price"
]

preprocessor = ColumnTransformer(

    transformers=[

        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),

        (
            "num",
            "passthrough",
            numeric_features
        )

    ]

)

model = CatBoostRegressor(

    iterations=700,
    depth=6,
    learning_rate=0.06,
    l2_leaf_reg=8,
    random_strength=2,
    bagging_temperature=1,
    subsample=0.85,
    loss_function="RMSE",
    random_seed=42,
    verbose=False

)

pipeline = Pipeline([

    ("preprocessor", preprocessor),

    ("model", model)

])

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.20,
    random_state=42

)

pipeline.fit(X_train, y_train)

train_pred = pipeline.predict(X_train)
test_pred = pipeline.predict(X_test)

In [7]:
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

mae = mean_absolute_error(y_test, test_pred)
rmse = np.sqrt(mean_squared_error(y_test, test_pred))

cv_scores = cross_val_score(

    pipeline,
    X,
    y,
    cv=5,
    scoring="r2",
    n_jobs=-1

)

print("\n===============================")
print("PRODUCTION TRAINING COMPLETE")
print("===============================\n")

print(f"Training R² : {train_r2:.4f}")
print(f"Testing R²  : {test_r2:.4f}")

print()

print(f"MAE  : ₹{mae:.2f}")
print(f"RMSE : ₹{rmse:.2f}")

print()

print("Cross Validation R² Scores")
print(cv_scores)

print()

print(f"Average CV R² : {cv_scores.mean():.4f}")

joblib.dump(
    pipeline,
    "clean_agri_model_new.pkl"
)

print("\nModel Saved : clean_agri_model_new.pkl")


PRODUCTION TRAINING COMPLETE

Training R² : 0.9081
Testing R²  : 0.8717

MAE  : ₹312.59
RMSE : ₹537.21

Cross Validation R² Scores
[0.8611355  0.85856152 0.84252844 0.90088547 0.82118483]

Average CV R² : 0.8569

Model Saved : clean_agri_model_new.pkl
